# 04 - Cohort Retention Analysis
## Measuring Customer Retention & Drop-Off Over Time

**Objective:**
Segment customers into monthly acquisition cohorts based on their first purchase date.
Track their retention curve over subsequent months and build the cohort heatmap.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PROCESSED_DIR = Path('../data/processed')
master = pd.read_csv(PROCESSED_DIR / 'master_analytical_orders.csv')
delivered = master[master['order_status'] == 'delivered'].copy()
delivered['order_purchase_timestamp'] = pd.to_datetime(delivered['order_purchase_timestamp'])

# Filter operational period 2017 to mid 2018
cohort_df = delivered[(delivered['order_purchase_timestamp'] >= '2017-01-01') & 
                      (delivered['order_purchase_timestamp'] <= '2018-08-31')].copy()


### 1. Assign Cohort Month and Order Index


In [ ]:
cohort_df['order_month'] = cohort_df['order_purchase_timestamp'].dt.to_period('M')
cohort_df['cohort_month'] = cohort_df.groupby('customer_unique_id')['order_month'].transform('min')

def get_cohort_index(df):
    return (df['order_month'].dt.year - df['cohort_month'].dt.year) * 12 + (df['order_month'].dt.month - df['cohort_month'].dt.month)

cohort_df['cohort_index'] = get_cohort_index(cohort_df)
print(cohort_df[['customer_unique_id', 'cohort_month', 'order_month', 'cohort_index']].head())


### 2. Build Cohort Retention Matrix


In [ ]:
cohort_counts = cohort_df.groupby(['cohort_month', 'cohort_index'])['customer_unique_id'].nunique().unstack()
cohort_sizes = cohort_counts.iloc[:, 0]
retention_matrix = cohort_counts.divide(cohort_sizes, axis=0) * 100
retention_matrix.round(2).head(10)


### 3. Plot Cohort Retention Heatmap


In [ ]:
plt.figure(figsize=(15, 8))
sns.heatmap(
    retention_matrix.iloc[1:18, :12],
    annot=True,
    fmt='.2f',
    cmap='Blues',
    linewidths=0.5,
    cbar_kws={'label': 'Retention Rate (%)'},
    vmin=0.0,
    vmax=1.2
)
plt.title('Customer Cohort Retention Rate (%) Across Operational Months', fontsize=15, fontweight='bold')
plt.xlabel('Months Since Acquisition (Cohort Index)')
plt.ylabel('Cohort Acquisition Month')
plt.show()
